# HybridBCIModel XAI Analysis
**DeepSHAP + Grad-CAM + ERD Validation — 52 Subjects (Cho2017 / GigaDB)**

| 항목 | 값 |
|---|---|
| 모델 | HybridBCIModel (EEGNet + BiLSTM + SoftmaxFusion) |
| 데이터 | GigaDB 100295, 52명 LOSO |
| 성능 | 74.2% ± 11.1% |
| 런타임 | A100 GPU 권장 |

**실행 순서:** 셀 하나씩 위에서 아래로 실행하세요.

---
## 0. 환경 설정

In [1]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 패키지 설치
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('shap')          # DeepSHAP
pip('mne')           # (optional) topomap rendering
print('설치 완료')

Mounted at /content/drive
설치 완료


In [2]:
# ────────────────────────────────────────────────────
#  ★ 여기만 수정하세요 ★
DRIVE_ROOT = '/content/drive/MyDrive/BCI_Research'
# ────────────────────────────────────────────────────

from pathlib import Path
import os

ROOT      = Path(DRIVE_ROOT)
DATA_DIR  = ROOT / 'preprocessed' / 'member_A'
CKPT_DIR  = ROOT / 'results' / 'checkpoints_A'
ACC_CSV   = ROOT / 'results' / 'ablation' / 'ablation_results.csv'
OUT_SHAP  = ROOT / 'results' / 'xai_shap'
OUT_GCAM  = ROOT / 'results' / 'xai_gradcam'
OUT_ERD   = ROOT / 'results' / 'xai_erd'
GCAM_DIR  = ROOT / 'results' / 'gradcam'

for d in [OUT_SHAP, OUT_GCAM, OUT_ERD, GCAM_DIR,
          OUT_SHAP/'figures', OUT_GCAM/'figures', OUT_ERD/'figures']:
    d.mkdir(parents=True, exist_ok=True)

print('경로 확인:')
print(f'  Data:  {DATA_DIR}  — 존재: {DATA_DIR.exists()}')
print(f'  Ckpt:  {CKPT_DIR}  — 존재: {CKPT_DIR.exists()}')
h5_files = list(DATA_DIR.glob('sub-*.h5'))
pt_files = list(CKPT_DIR.glob('best_s*.pt'))
print(f'  HDF5: {len(h5_files)}개 | Checkpoint: {len(pt_files)}개')

경로 확인:
  Data:  /content/drive/MyDrive/BCI_Research/preprocessed/member_A  — 존재: True
  Ckpt:  /content/drive/MyDrive/BCI_Research/results/checkpoints_A  — 존재: True
  HDF5: 52개 | Checkpoint: 52개


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import hilbert, spectrogram
from scipy.stats import wilcoxon
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import griddata

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}  |  Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. 모델 정의 (공통)

In [ ]:
CFG = {
    'n_eeg_ch': 64, 'n_emg_ch': 4, 'n_times': 2304, 'n_classes': 2,
    'emg_ds_factor': 8, 'eegnet_F1': 8, 'eegnet_D': 2, 'eegnet_kern_len': 256,
    'eegnet_dropout': 0.5, 'lstm_hidden': 128, 'lstm_layers': 2,
    'lstm_dropout': 0.3, 'clf_dropout': 0.3, 'feat_dim': 256,
}
CFG['n_times_emg'] = CFG['n_times'] // CFG['emg_ds_factor']

class EEGNetEncoder(nn.Module):
    def __init__(self, n_ch, n_times, F1=8, D=2, kern_len=256, dropout=0.5, feat_dim=256):
        super().__init__()
        F2 = F1 * D
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kern_len), padding=(0, kern_len//2), bias=False),
            nn.BatchNorm2d(F1))
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F2, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dropout))
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, 1, bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(dropout))
        flat = self._flat(n_ch, n_times)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(flat, feat_dim), nn.ELU())

    def _flat(self, n_ch, n_times):
        with torch.no_grad():
            return self.block3(self.block2(self.block1(torch.zeros(1, 1, n_ch, n_times)))).numel()

    def forward(self, x):
        return self.fc(self.block3(self.block2(self.block1(x.unsqueeze(1)))))

class EMGBiLSTMEncoder(nn.Module):
    def __init__(self, n_ch=4, hidden=128, n_layers=2, dropout=0.3, feat_dim=256):
        super().__init__()
        self.lstm = nn.LSTM(n_ch, hidden, n_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden * 2)
        self.fc   = nn.Sequential(nn.Linear(hidden * 2, feat_dim), nn.ELU())
    def forward(self, x):
        out, _ = self.lstm(x.permute(0, 2, 1))
        return self.fc(self.norm(out[:, -1, :]))

class SoftmaxAttentionFusion(nn.Module):
    def __init__(self, feat_dim=256):
        super().__init__()
        self.W_eeg = nn.Linear(feat_dim, feat_dim)
        self.W_emg = nn.Linear(feat_dim, feat_dim)
        self.attn  = nn.Linear(feat_dim * 2, 2)
    def forward(self, h_eeg, h_emg):
        w = F.softmax(self.attn(torch.cat([h_eeg, h_emg], dim=-1)), dim=-1)
        return w[:, 0:1]*self.W_eeg(h_eeg) + w[:, 1:2]*self.W_emg(h_emg), w

class HybridBCIModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        fd = cfg['feat_dim']
        self.eeg_enc = EEGNetEncoder(cfg['n_eeg_ch'], cfg['n_times'],
            cfg['eegnet_F1'], cfg['eegnet_D'], cfg['eegnet_kern_len'],
            cfg['eegnet_dropout'], fd)
        self.emg_enc = EMGBiLSTMEncoder(cfg['n_emg_ch'], cfg['lstm_hidden'],
            cfg['lstm_layers'], cfg['lstm_dropout'], fd)
        self.fusion  = SoftmaxAttentionFusion(fd)
        self.clf     = nn.Sequential(nn.Linear(fd, 128), nn.ELU(),
                                     nn.Dropout(cfg['clf_dropout']),
                                     nn.Linear(128, cfg['n_classes']))
    def forward(self, eeg, emg):
        fused, w = self.fusion(self.eeg_enc(eeg), self.emg_enc(emg))
        return self.clf(fused), w

class _LogitOnly(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, eeg, emg): return self.m(eeg, emg)[0]

def load_model(sid, device=DEVICE):
    ckpt = CKPT_DIR / f'best_s{sid:02d}.pt'
    m = HybridBCIModel(CFG).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    return m

def load_data(sid):
    ds = CFG['emg_ds_factor']
    with h5py.File(DATA_DIR / f'sub-{sid:02d}_member_A.h5', 'r') as f:
        eeg = f['eeg/epochs'][:].astype(np.float32)
        emg = f['emg/epochs'][:, :, ::ds].astype(np.float32)
        lbl = f['labels'][:].astype(np.int64) - 1
    return eeg, emg, lbl

# 64-channel names (C3=18, Cz=20, C4=22)
CH_NAMES = [
    'Fp1','AF7','AF3','F1', 'F3', 'F5', 'F7',
    'FT7','FC5','FC3','FC1','FCz','FC2','FC4',
    'FC6','FT8','T7', 'C5', 'C3', 'C1', 'Cz',
    'C2', 'C4', 'C6', 'T8', 'TP7','CP5','CP3',
    'CP1','CPz','CP2','CP4','CP6','TP8',
    'P7', 'P5', 'P3', 'P1', 'Pz', 'P2',
    'P4', 'P6', 'P8', 'PO7','PO3','POz',
    'PO4','PO8','O1', 'Oz', 'O2', 'Iz',
    'Fp2','AF8','AF4','F2', 'F4', 'F6', 'F8',
    'FT9','FT10','TP9','TP10','Fpz',
]
CH_IDX = {ch: i for i, ch in enumerate(CH_NAMES)}
MOTOR_CHS = ['C3','C1','Cz','C2','C4','FC3','FC1','FCz','FC2','FC4',
              'CP3','CP1','CPz','CP2','CP4']
SIDS = list(range(1, 53))

print('모델 정의 완료  |  C3 index:', CH_IDX['C3'], ' C4 index:', CH_IDX['C4'])
# Quick sanity: load s01
_m = load_model(1); _m.eval()
with torch.no_grad():
    _eeg, _emg, _ = load_data(1)
    _out, _ = _m(torch.tensor(_eeg[:2]).to(DEVICE), torch.tensor(_emg[:2]).to(DEVICE))
print(f'모델 테스트 OK  |  출력 shape: {_out.shape}  softmax: {F.softmax(_out, -1)[0].cpu().numpy()}')
del _m, _eeg, _emg, _out

---
## 2. DeepSHAP — 52명 Feature Importance

각 피험자의 LOSO 모델로 EEG(64ch) + sEMG(4ch) SHAP 중요도 계산.
- Background: 각 피험자 100샘플 (50 Left + 50 Right)
- 예상 소요시간: ~2분/피험자 → 52명 약 90분 (A100)

In [ ]:
import shap
print(f'shap {shap.__version__}')

N_BG   = 100   # background 샘플 수 (줄이면 빠르지만 정확도 하락)
N_TEST = 200   # test 샘플 수 (=전체)

shap_records = []

for sid in SIDS:
    h5_path   = DATA_DIR / f'sub-{sid:02d}_member_A.h5'
    ckpt_path = CKPT_DIR / f'best_s{sid:02d}.pt'
    if not h5_path.exists() or not ckpt_path.exists():
        print(f'[s{sid:02d}] 파일 없음 — 건너뜀'); continue

    eeg, emg, lbl = load_data(sid)
    model   = load_model(sid).eval()
    wrapper = _LogitOnly(model).eval()

    rng  = np.random.RandomState(42 + sid)
    idx0 = np.where(lbl == 0)[0];  idx1 = np.where(lbl == 1)[0]
    bg_idx = np.concatenate([rng.choice(idx0, N_BG//2, replace=False),
                              rng.choice(idx1, N_BG//2, replace=False)])
    te_idx = np.setdiff1d(rng.permutation(len(lbl))[:N_TEST], bg_idx)

    bg_eeg = torch.tensor(eeg[bg_idx], device=DEVICE)
    bg_emg = torch.tensor(emg[bg_idx], device=DEVICE)
    te_eeg = torch.tensor(eeg[te_idx], device=DEVICE)
    te_emg = torch.tensor(emg[te_idx], device=DEVICE)

    try:
        explainer = shap.DeepExplainer(wrapper, [bg_eeg, bg_emg])
        sv = explainer.shap_values([te_eeg, te_emg])
        # sv: list(2 classes) of list(2 inputs)
        eeg_imp = np.stack([np.abs(sv[c][0]).mean((0,2)) for c in range(2)])  # (2,64)
        emg_imp = np.stack([np.abs(sv[c][1]).mean((0,2)) for c in range(2)])  # (2,4)

        with torch.no_grad():
            logits = wrapper(te_eeg, te_emg).cpu().numpy()
        acc = float((logits.argmax(1) == lbl[te_idx]).mean())

        shap_records.append({'sid': sid, 'eeg_imp': eeg_imp,
                              'emg_imp': emg_imp, 'acc': acc})
        top_ch = CH_NAMES[eeg_imp.mean(0).argmax()]
        print(f'[s{sid:02d}] ✓ acc={acc:.3f} top_EEG={top_ch}')
    except Exception as e:
        print(f'[s{sid:02d}] ✗ {e}')

print(f'\n완료: {len(shap_records)}/52 피험자')

In [ ]:
# 52명 집계
if not shap_records:
    n_missing = sum(
        1 for sid in SIDS
        if not (DATA_DIR / f'sub-{sid:02d}_member_A.h5').exists()
        or not (CKPT_DIR / f'best_s{sid:02d}.pt').exists()
    )
    raise RuntimeError(
        f"shap_records가 비어 있습니다 — 위 셀(shap_run) 출력을 확인하세요.\n"
        f"  파일 없는 피험자: {n_missing}/{len(SIDS)}명\n"
        f"  DATA_DIR 존재: {DATA_DIR.exists()}  ({DATA_DIR})\n"
        f"  CKPT_DIR 존재: {CKPT_DIR.exists()}  ({CKPT_DIR})\n"
        f"해결: 경로가 맞다면 SHAP 예외 메시지(✗)를 확인하고 해당 오류를 먼저 수정하세요."
    )

eeg_mean = np.stack([r['eeg_imp'] for r in shap_records]).mean(0)  # (2,64)
eeg_std  = np.stack([r['eeg_imp'] for r in shap_records]).std(0)
emg_mean = np.stack([r['emg_imp'] for r in shap_records]).mean(0)  # (2,4)
emg_std  = np.stack([r['emg_imp'] for r in shap_records]).std(0)

# NPZ 저장
np.savez_compressed(OUT_SHAP/'shap_channel_importance.npz',
    eeg_mean=eeg_mean, eeg_std=eeg_std,
    emg_mean=emg_mean, emg_std=emg_std,
    ch_names=np.array(CH_NAMES))

# Per-subject CSV
rows = []
for r in shap_records:
    for c, cls in enumerate(['Left_MI','Right_MI']):
        top5 = np.argsort(r['eeg_imp'][c])[::-1][:5]
        rows.append({'sid': r['sid'], 'class': cls,
                     **{f'top{k+1}': CH_NAMES[i] for k,i in enumerate(top5)},
                     'c3_shap': r['eeg_imp'][c][CH_IDX['C3']],
                     'c4_shap': r['eeg_imp'][c][CH_IDX['C4']],
                     'acc': r['acc']})
pd.DataFrame(rows).to_csv(OUT_SHAP/'shap_per_subject.csv', index=False)

print('저장 완료')
for c, cls in enumerate(['Left MI','Right MI']):
    top = np.argsort(eeg_mean[c])[::-1][:5]
    print(f'[{cls}] Top-5: {[CH_NAMES[i] for i in top]}')
    print(f'  C3={eeg_mean[c][CH_IDX["C3"]]:.4f}  '
          f'Cz={eeg_mean[c][CH_IDX["Cz"]]:.4f}  '
          f'C4={eeg_mean[c][CH_IDX["C4"]]:.4f}')

In [ ]:
from matplotlib.patches import Patch
motor_set = set(MOTOR_CHS)

# ── Figure A: Top-20 EEG 채널 중요도 (Left / Right MI)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for c, (ax, cls) in enumerate(zip(axes, ['Left MI','Right MI'])):
    idx = np.argsort(eeg_mean[c])[::-1][:20]
    color = ['#4CAF50' if CH_NAMES[i] in motor_set else '#9E9E9E' for i in idx]
    ax.barh(range(20), eeg_mean[c][idx][::-1], xerr=eeg_std[c][idx][::-1],
            color=color[::-1], alpha=0.85, ecolor='black', capsize=3)
    ax.set_yticks(range(20))
    ax.set_yticklabels([CH_NAMES[i] for i in idx][::-1], fontsize=9)
    ax.set_xlabel('Mean |SHAP|'); ax.set_title(f'{cls} — Top-20 EEG Channels', fontweight='bold')
    ax.legend(handles=[Patch(color='#4CAF50',label='Motor cortex'),
                        Patch(color='#9E9E9E',label='Other')], fontsize=8)
fig.suptitle(f'EEG Feature Importance — DeepSHAP (n={len(shap_records)})', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_SHAP/'figures'/'fig_eeg_shap_bar.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_SHAP/'figures'/'fig_eeg_shap_bar.pdf', bbox_inches='tight')
plt.show()

# ── Figure B: sEMG 채널 중요도
EMG_NAMES = ['EMG1','EMG2','EMG3','EMG4']
fig, ax = plt.subplots(figsize=(6,4))
x = np.arange(4); w = 0.35
for c, (cls, col) in enumerate(zip(['Left MI','Right MI'],['#2196F3','#F44336'])):
    ax.bar(x+c*w, emg_mean[c], w, yerr=emg_std[c], label=cls,
           color=col, alpha=0.8, ecolor='black', capsize=4)
ax.set_xticks(x+w/2); ax.set_xticklabels(EMG_NAMES)
ax.set_ylabel('Mean |SHAP|'); ax.legend()
ax.set_title(f'sEMG Feature Importance — DeepSHAP (n={len(shap_records)})', fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_SHAP/'figures'/'fig_emg_shap.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Figure C: Laterality (C3 vs C4)
fig, ax = plt.subplots(figsize=(7,4))
c3i, c4i = CH_IDX['C3'], CH_IDX['C4']
for c, (cls, col) in enumerate(zip(['Left MI','Right MI'],['#2196F3','#F44336'])):
    ax.bar([c*3, c*3+1],
           [eeg_mean[c][c3i], eeg_mean[c][c4i]],
           yerr=[eeg_std[c][c3i], eeg_std[c][c4i]],
           color=[col,col], alpha=[0.9,0.5], ecolor='black', capsize=4)
ax.set_xticks([0,1,3,4])
ax.set_xticklabels(['C3\n(Left)','C4\n(Right)','C3\n(Left)','C4\n(Right)'])
ax.set_ylabel('Mean |SHAP|')
ax.set_title('Hemispheric Laterality of SHAP Importance\nLeft MI: C4↑ | Right MI: C3↑ (contralateral)', fontweight='bold')
ax.legend(handles=[Patch(color='#2196F3',label='Left MI'), Patch(color='#F44336',label='Right MI')])
plt.tight_layout()
fig.savefig(OUT_SHAP/'figures'/'fig_laterality.png', dpi=300, bbox_inches='tight')
plt.show()

print('DeepSHAP 피규어 저장 완료')

---
## 4. ERD Validation — Alpha/Beta Band Desynchronization

HDF5에 사전 저장된 밴드 필터링 신호 (`eeg/mu_epochs`, `eeg/beta_epochs`) 활용.
- ERD(t) = 10·log10[ P(t) / P_baseline ] (dB)
- 음수 dB = 탈동기화 (Motor Imagery 중 예상)
- GPU 불필요 (scipy만 사용)

In [ ]:
FS = 512; TMIN = -0.5; TMAX = 4.0; N_T = 2304
T  = np.linspace(TMIN, TMAX, N_T)
BL_MASK = (T >= -0.5) & (T < 0.0)   # 기저선 창
MI_MASK = (T >= 0.5)  & (T < 3.0)   # MI 평가 창

def inst_pow(sig): return np.abs(hilbert(sig, axis=-1))**2

def compute_erd(sig, bl_mask):
    """ERD in dB; Input: (N,ch,T), Output: (ch,T)"""
    p   = inst_pow(sig)
    pbl = p[:,:,bl_mask].mean(-1,keepdims=True)
    return (10*np.log10(p/(pbl+1e-12))).mean(0)

erd_records = []
for sid in SIDS:
    h5 = DATA_DIR/f'sub-{sid:02d}_member_A.h5'
    if not h5.exists(): continue
    with h5py.File(h5,'r') as f:
        mu   = f['eeg/mu_epochs'][:].astype(np.float32)
        beta = f['eeg/beta_epochs'][:].astype(np.float32)
        lbl  = f['labels'][:].astype(np.int64) - 1
    il, ir = np.where(lbl==0)[0], np.where(lbl==1)[0]
    erd_records.append({
        'sid': sid,
        'mu_l':   compute_erd(mu[il],   BL_MASK),
        'mu_r':   compute_erd(mu[ir],   BL_MASK),
        'beta_l': compute_erd(beta[il], BL_MASK),
        'beta_r': compute_erd(beta[ir], BL_MASK),
    })

n = len(erd_records)
print(f'ERD 계산 완료: {n}명')

def grp(key): return {'mean': np.stack([r[key] for r in erd_records]).mean(0),
                      'sem':  np.stack([r[key] for r in erd_records]).std(0)/np.sqrt(n)}
GRP = {k: grp(k) for k in ['mu_l','mu_r','beta_l','beta_r']}

# 핵심 채널 리포트
c3i, czi, c4i = CH_IDX['C3'], CH_IDX['Cz'], CH_IDX['C4']
print('\n채널별 ERD dB (MI window 0.5–3.0s, 그룹 평균):')
print(f'       {'':8}  {'Left MI':>12}  {'Right MI':>12}')
for ch, idx in [('C3',c3i),('Cz',czi),('C4',c4i)]:
    for band, lkey, rkey in [('Mu',  'mu_l', 'mu_r'),
                               ('Beta','beta_l','beta_r')]:
        l = GRP[lkey]['mean'][idx,MI_MASK].mean()
        r = GRP[rkey]['mean'][idx,MI_MASK].mean()
        print(f'  {ch} {band}: {l:+.2f} dB          {r:+.2f} dB')

In [ ]:
# Wilcoxon 통계
rows=[]
for ch,idx in [('C3',c3i),('C4',c4i),('Cz',czi)]:
    for band,lk,rk in [('mu','mu_l','mu_r'),('beta','beta_l','beta_r')]:
        for cls,key in [('left',lk),('right',rk)]:
            vals = np.array([r[key][idx,MI_MASK].mean() for r in erd_records])
            try: _,p = wilcoxon(vals, alternative='less')
            except: p=np.nan
            sig='***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else 'ns'))
            rows.append({'channel':ch,'band':band,'class':cls,
                         'mean_erd':round(vals.mean(),3),'sd':round(vals.std(),3),
                         'p':round(float(p),4),'sig':sig,'n':n})

sig_df = pd.DataFrame(rows)
sig_df.to_csv(OUT_ERD/'erd_significance.csv', index=False)

# Laterality Index
lat_rows=[]
for r in erd_records:
    for band,lk,rk in [('mu','mu_l','mu_r'),('beta','beta_l','beta_r')]:
        lat_rows.append({'sid':r['sid'],'band':band,
            'LI_left': float(r[lk][c4i,MI_MASK].mean()-r[lk][c3i,MI_MASK].mean()),
            'LI_right':float(r[rk][c3i,MI_MASK].mean()-r[rk][c4i,MI_MASK].mean())})
lat_df = pd.DataFrame(lat_rows)
lat_df.to_csv(OUT_ERD/'erd_laterality.csv', index=False)

print('=== Wilcoxon (C3 / C4 ERD < 0 dB) ===')
print(sig_df[sig_df['channel'].isin(['C3','C4'])].to_string(index=False))
print('\n=== Laterality Index (ERD_contra - ERD_ipsi) ===')
for band in ['mu','beta']:
    d=lat_df[lat_df['band']==band]
    for k,cls in [('LI_left','Left MI'),('LI_right','Right MI')]:
        v=d[k].values
        _,p=wilcoxon(v,alternative='less') if len(v)>1 else (0,1)
        sig='***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else 'ns'))
        print(f'  {band:4s} {cls}: LI={v.mean():+.2f}±{v.std():.2f} dB  {sig}')

In [ ]:
# ── Figure A: ERD 시간 경과 (C3, Cz, C4)
fig,axes = plt.subplots(2,2,figsize=(12,7),sharex=True)
styles={'C3':'-','Cz':'--','C4':':'}
for row,(band,lk,rk) in enumerate([('Mu (8–12 Hz)','mu_l','mu_r'),
                                     ('Beta (13–30 Hz)','beta_l','beta_r')]):
    for col,(cls,key,color) in enumerate([('Left MI',lk,'#2196F3'),
                                           ('Right MI',rk,'#F44336')]):
        ax=axes[row,col]
        for ch,st in styles.items():
            m=GRP[key]['mean'][CH_IDX[ch]]
            s=GRP[key]['sem'][CH_IDX[ch]]
            ax.plot(T,m,st,color=color,lw=1.5,label=ch)
            ax.fill_between(T,m-s,m+s,color=color,alpha=.15)
        ax.axhline(0,color='gray',lw=.8); ax.axvline(0,color='k',lw=1,ls='--',alpha=.7)
        ax.axvspan(.5,3.,alpha=.06,color='green')
        ax.set_xlim(TMIN,TMAX)
        ax.set_title(f'{band} — {cls}',fontweight='bold',fontsize=10)
        if col==0: ax.set_ylabel('ERD (dB)',fontsize=9)
        if row==1: ax.set_xlabel('Time (s)',fontsize=9)
        if row==0 and col==0: ax.legend(fontsize=8)
fig.suptitle(f'ERD Time Course — Group Average (n={n})\nNegative dB = desynchronization',
             fontsize=12,fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_ERD/'figures'/'fig_erd_timecourse.png',dpi=300,bbox_inches='tight')
fig.savefig(OUT_ERD/'figures'/'fig_erd_timecourse.pdf',bbox_inches='tight')
plt.show()

# ── Figure B: Laterality box plot
fig,axes=plt.subplots(1,2,figsize=(10,5))
for ax,band in zip(axes,['mu','beta']):
    db=lat_df[lat_df['band']==band]
    for i,(k,cls,col) in enumerate([('LI_left','Left MI','#2196F3'),
                                      ('LI_right','Right MI','#F44336')]):
        v=db[k].values
        ax.scatter(np.full(len(v),i)+np.random.uniform(-.1,.1,len(v)),
                   v,alpha=.4,s=20,color=col)
        ax.boxplot(v,positions=[i],widths=.35,patch_artist=True,
                   boxprops=dict(facecolor=col,alpha=.3),
                   medianprops=dict(color='black',linewidth=2))
    ax.axhline(0,color='gray',lw=1,ls='--')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Left MI','Right MI'],fontsize=10)
    bstr='Mu (8–12 Hz)' if band=='mu' else 'Beta (13–30 Hz)'
    ax.set_title(f'{bstr}\nLI = ERD_contra − ERD_ipsi (dB)',fontweight='bold')
    ax.set_ylabel('Laterality Index (dB)')
fig.suptitle(f'Contralateral ERD Laterality (n={n})\nNegative expected for correct lateralization',
             fontsize=12,fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_ERD/'figures'/'fig_erd_laterality.png',dpi=300,bbox_inches='tight')
plt.show()

print('ERD 피규어 저장 완료')

In [ ]:
# ── Figure C: STFT 시간-주파수 스펙트로그램 (C3, C4)
MAX_TF_SUBS = 20   # 메모리 제한
raw_l, raw_r = [], []
for r in erd_records[:MAX_TF_SUBS]:
    with h5py.File(DATA_DIR/f'sub-{r["sid"]:02d}_member_A.h5','r') as f:
        raw  = f['eeg/epochs'][:].astype(np.float32)
        lbl2 = f['labels'][:].astype(np.int64)-1
    raw_l.append(raw[lbl2==0])
    raw_r.append(raw[lbl2==1])

def tf_erd(trials_ch):
    """trials_ch: (N,T) for one channel; returns freqs, t_ep, ERD_TF"""
    stacks=[]
    for tr in trials_ch:
        f,t,S = spectrogram(tr,fs=FS,nperseg=256,noverlap=224,scaling='density')
        stacks.append(S)
    S_mean=np.stack(stacks).mean(0)
    bl_end=-TMIN  # 0.5s in signal time
    bl=t<bl_end;  pbl=S_mean[:,bl].mean(-1,keepdims=True) if bl.any() else S_mean[:,:1]
    return f, t+TMIN, (S_mean-pbl)/(pbl+1e-12)*100

fig,axes=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
for row,(cls,raw_list) in enumerate([('Left MI',raw_l),('Right MI',raw_r)]):
    for col,(ch,idx) in enumerate([('C3',c3i),('C4',c4i)]):
        trials=np.concatenate([r[:,idx,:] for r in raw_list])
        if len(trials)>300: trials=trials[np.random.choice(len(trials),300,replace=False)]
        freqs,times,erd_tf=tf_erd(trials)
        fm=(freqs>=4)&(freqs<=40)
        ax=axes[row,col]
        im=ax.pcolormesh(times,freqs[fm],erd_tf[fm],cmap='RdBu_r',
                          vmin=-100,vmax=100,shading='gouraud')
        ax.axvline(0,color='white',lw=1.5,ls='--')
        for hz,col2 in [(8,'white'),(12,'white'),(15,'yellow'),(30,'yellow')]:
            ax.axhline(hz,color=col2,lw=.8,ls=':',alpha=.8)
        ax.set_xlim(TMIN,TMAX); ax.set_ylim(4,40)
        ax.set_title(f'{cls} — {ch}',fontweight='bold',fontsize=10)
        ax.set_xlabel('Time (s)',fontsize=9)
        if col==0: ax.set_ylabel('Frequency (Hz)',fontsize=9)
        plt.colorbar(im,ax=ax,shrink=.8).set_label('ERD (%)',fontsize=8)
fig.suptitle(f'Time-Frequency ERD (n={MAX_TF_SUBS} subj sample)\n'
             'White: μ band (8-12Hz) | Yellow: β band (15-30Hz)',
             fontsize=12,fontweight='bold')
fig.savefig(OUT_ERD/'figures'/'fig_erd_spectrogram.png',dpi=300,bbox_inches='tight')
fig.savefig(OUT_ERD/'figures'/'fig_erd_spectrogram.pdf',bbox_inches='tight')
plt.show()
print('스펙트로그램 저장 완료')

---
## 5. 결과 요약

In [ ]:
print('='*65)
print('  S4 XAI Analysis 결과 요약 (JNE 심사용)')
print('='*65)

print(f'\n[1] DeepSHAP  — {len(shap_records)}명 완료')
for c,cls in enumerate(['Left MI','Right MI']):
    top3 = [CH_NAMES[i] for i in np.argsort(eeg_mean[c])[::-1][:3]]
    print(f'   {cls}: Top-3 EEG = {top3}')
    print(f'      C3={eeg_mean[c][c3i]:.4f}  '
          f'Cz={eeg_mean[c][czi]:.4f}  '
          f'C4={eeg_mean[c][c4i]:.4f}')

print(f'\n[2] Grad-CAM  — {len(gcam_records)}명 완료')
for key,cls in [('left_mi','Left MI'),('right_mi','Right MI')]:
    avg = np.mean([r[key] for r in gcam_records],0)
    top3=[CH_NAMES[i] for i in np.argsort(avg)[::-1][:3]]
    print(f'   {cls}: Top-3 = {top3}')

print(f'\n[3] ERD Validation  — {n}명 완료')
motor_rows = sig_df[sig_df['channel'].isin(['C3','C4'])]
print(motor_rows[['channel','band','class','mean_erd','sd','p','sig']].to_string(index=False))

print('\n[4] Laterality Index')
for band in ['mu','beta']:
    d=lat_df[lat_df['band']==band]
    for k,cls in [('LI_left','Left MI'),('LI_right','Right MI')]:
        v=d[k].values
        print(f'   {band:4s} {cls}: LI={v.mean():+.2f}±{v.std():.2f} dB')

print('\n[5] 저장 위치')
print(f'   SHAP:     {OUT_SHAP}')
print(f'   Grad-CAM: {GCAM_DIR} (figure6용 NPZ)')
print(f'   ERD:      {OUT_ERD}')

print('\n[6] 다음 단계')
print('   !pip install mne -q')
print(f'   !python src/figure6_gradcam_topomaps.py \\')
print(f'     --gradcam-dir {GCAM_DIR} \\')
print(f'     --positions-csv {GCAM_DIR}/channel_positions.csv \\')
print(f'     --accuracy-csv {ACC_CSV} \\')
print(f'     --output-dir {OUT_GCAM}/figure6')
print('='*65)